In [ ]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
import numpy as np
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
from common_lib.segmentation import (
    add_order_prefix,
    add_vlines_to_figure,
    add_segment_transition_columns,
    get_segment_movements_agg,
    plot_movement_charts,
    get_segment_population_agg,
    plot_segment_population_charts,
    get_segment_performance_summary,
    plot_segment_performance_summary,
)
import datetime as dt
import plotly.express as px

In [2]:
query_location = './sql/activity.sql'
parameters = {
    'start_date':'2026-01-01',
    'end_date': dt.datetime.now().strftime('%Y-%m-%d'),
    'exclude_networks':['']
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 8.60 GB when run.
Estimated query cost: $0.06


In [3]:
refresh_data = False

In [4]:
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query='./sql/activity.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/activity.pkl')
else:
    data = pd.read_pickle('./data/activity.pkl')


In [5]:

data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,loyalty_segment,payer_value_segment,payment_frequency_segment,payer_recency_segment,dsi_segment,daily_offers_avg_segment,daily_offers_count_segment,daily_offers_lastpurchase_segment,usd_net_iap_revenue,usd_net_ad_revenue
0,64BD21CBFD32BB1B,2026-01-02,2025-12-28,2026-01-01,2025-02-13,2025-02-09,2025-02-01,323,1. 26-28 (dedicated),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D182-D363,0.00,0,none,NaN,0.140343
1,7B0C1FBC2B8876AF,2026-01-02,2025-12-28,2026-01-01,2025-11-24,2025-11-23,2025-11-01,39,2. 19-25 (frequent),0.1 lapsed_payer,0.1 lapsed_payer,0.1 lapsed_payer,D028-D090,0.00,0,none,NaN,0.054277
2,E2B523EB63F93C25,2026-01-03,2025-12-28,2026-01-01,2025-11-06,2025-11-02,2025-11-01,58,1. 26-28 (dedicated),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D028-D090,0.00,0,none,NaN,0.006553
3,4042C6F11F2451BF,2026-01-01,2025-12-28,2026-01-01,2025-10-15,2025-10-12,2025-10-01,78,1. 26-28 (dedicated),0.1 lapsed_payer,0.1 lapsed_payer,0.1 lapsed_payer,D028-D090,0.00,0,none,NaN,0.265574
4,DD05B38FB3C5E0B1,2026-01-03,2025-12-28,2026-01-01,2022-06-05,2022-06-05,2022-06-01,1308,1. 26-28 (dedicated),5.$80.00-$139.99,4.12-22,2.03-07,D364+,8.00_14.99,2_9,0d_1d,11.095804,0.065430
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20796291,A81FE1492C9FE7A4,2026-08-18,2026-08-16,2026-08-01,2026-02-16,2026-02-15,2026-02-01,183,1. 26-28 (dedicated),0.1 lapsed_payer,0.1 lapsed_payer,0.1 lapsed_payer,D182-D363,0.00,0,none,NaN,0.077350
20796292,481DA1902AC8A52,2026-08-16,2026-08-16,2026-08-01,2022-10-21,2022-10-16,2022-10-01,1395,1. 26-28 (dedicated),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D364+,0.00,0,none,NaN,NaN
20796293,B2CD5E0017B230A0,2026-08-18,2026-08-16,2026-08-01,2022-02-08,2022-02-06,2022-02-01,1652,1. 26-28 (dedicated),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D364+,0.00,0,none,NaN,0.002489
20796294,9F6DB085566B127C,2026-08-17,2026-08-16,2026-08-01,2026-01-12,2026-01-11,2026-01-01,217,1. 26-28 (dedicated),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D182-D363,0.00,0,none,NaN,NaN


## Process data

In [6]:
data = add_order_prefix(data, ['daily_offers_avg_segment', 'daily_offers_count_segment', 'daily_offers_lastpurchase_segment'])
data[['daily_offers_avg_segment', 'daily_offers_count_segment', 'daily_offers_lastpurchase_segment']]

,daily_offers_avg_segment,daily_offers_count_segment,daily_offers_lastpurchase_segment
0,0_0.00,0_0,4_none
1,0_0.00,0_0,4_none
2,0_0.00,0_0,4_none
3,0_0.00,0_0,4_none
4,4_8.00_14.99,2_2_9,0_0d_1d
...,...,...,...
20796291,0_0.00,0_0,4_none
20796292,0_0.00,0_0,4_none
20796293,0_0.00,0_0,4_none
20796294,0_0.00,0_0,4_none


In [7]:
# Collapse "lapsed_payer" into the non-payer tier for all three payer segments — the SQL's own
# COALESCE default for a true never-paid user is '0.0 (non-payer)', and the dimension table's
# lapsed-payer label is '0.1 lapsed_payer' (identical string across all three columns), so a
# lapsed payer and a never-paid user end up on the same tier going forward. Flag columns keep
# the distinction available for anyone who wants it, done before the transition columns above
# so jump_size/duration treat lapsed-vs-non-payer as no shift, not a fake tier change.
LAPSED_LABEL = '0.1 lapsed_payer'
NON_PAYER_LABEL = '0.0 (non-payer)'

for col in ['payer_value_segment', 'payment_frequency_segment', 'payer_recency_segment']:
    data[f'{col}_is_lapsed'] = data[col] == LAPSED_LABEL
    data[col] = data[col].replace(LAPSED_LABEL, NON_PAYER_LABEL)




### Milestone dates

In [8]:
# Define vlines configuration
# Defines event timeline markers used for chart annotations
# Each event's line_color is optional (add_vlines_to_figure defaults to 'gray' if omitted)
vlines_events = [
    # Timed Albums
    #{'x': pd.Timestamp('2025-11-04').timestamp() * 1000, 'annotation_text': 'JoesTastyTravels'},
    #{'x': pd.Timestamp('2025-12-16').timestamp() * 1000, 'annotation_text': 'WinterTales'},
    #{'x': pd.Timestamp('2026-01-26').timestamp() * 1000, 'annotation_text': 'AppletonLove', 'line_color': 'brown'},
    #{'x': pd.Timestamp('2026-03-08').timestamp() * 1000, 'annotation_text': 'Diorama', 'line_color': 'brown'},
    #{'x': pd.Timestamp('2026-04-18').timestamp() * 1000, 'annotation_text': 'ThroughTheAges', 'line_color': 'brown'},
    #{'x': pd.Timestamp('2026-05-29').timestamp() * 1000, 'annotation_text': 'Appletonians', 'line_color': 'brown'},
    #{'x': pd.Timestamp('2026-07-07').timestamp() * 1000, 'annotation_text': 'BitesizedAdventures', 'line_color': 'brown'},
    #{'x': pd.Timestamp('2026-08-17').timestamp() * 1000, 'annotation_text': 'PlayTime', 'line_color': 'brown'},
    #APS7
    {'x': pd.Timestamp('2026-06-15').timestamp() * 1000,'annotation_text': 'Step1Start', 'line_color': 'black'},
    #{'x': pd.Timestamp('2026-06-19').timestamp() * 1000,'annotation_text': 'AdsIssue_CA_AU'},
    {'x': pd.Timestamp('2026-06-29').timestamp() * 1000,'annotation_text': 'Step2Applied', 'line_color': 'black'},
    #{'x': pd.Timestamp('2026-07-08').timestamp() * 1000,'annotation_text': 'v0.79Rollout'},
    {'x': pd.Timestamp('2026-07-14').timestamp() * 1000,'annotation_text': 'BackToStep1', 'line_color': 'black'},
    #{'x': pd.Timestamp('2026-07-16').timestamp() * 1000,'annotation_text': 'FcapTo15_AllPlayers'},
    #{'x': pd.Timestamp('2026-08-04').timestamp() * 1000,'annotation_text': 'FcapTo30_AllPlayers'},
]

## ARPDAU

In [9]:
data_arpdau_agg = data.groupby('dt').agg(
    usd_net_iap_revenue = ('usd_net_iap_revenue', 'sum'),
    usd_net_ad_revenue = ('usd_net_ad_revenue', 'sum'),
    user_id_count = ('user_id', 'nunique'),
).reset_index()

data_arpdau_agg['arpdau'] = (data_arpdau_agg['usd_net_iap_revenue'] + data_arpdau_agg['usd_net_ad_revenue']) / data_arpdau_agg['user_id_count'].replace(0, np.nan)
data_arpdau_agg

,dt,usd_net_iap_revenue,usd_net_ad_revenue,user_id_count,arpdau
0,2026-01-01,28755.059962,8882.558744,112469,0.334649
1,2026-01-02,44782.949123,9260.298230,116098,0.465497
2,2026-01-03,29171.675900,9704.887634,116528,0.333624
3,2026-01-04,24169.229485,10071.831181,118270,0.289516
4,2026-01-05,23864.662400,9528.951608,116350,0.287010
...,...,...,...,...,...
226,2026-08-15,17195.471099,4888.821777,61640,0.358279
227,2026-08-16,11814.066088,5012.197022,62348,0.269877
228,2026-08-17,12457.984035,5428.631085,62491,0.286227
229,2026-08-18,13789.840769,5714.682380,62711,0.311022


In [10]:
fig = px.line(data_arpdau_agg, 
              x='dt', 
              y=['arpdau'],
              #color='payer_value_segment',
              title='ARPDAU Over Time',
              width=1500,
              height=600,
              hover_data={'user_id_count': True, 'usd_net_iap_revenue': True, 'usd_net_ad_revenue': True},
              #barmode='group'
              )

fig = add_vlines_to_figure(fig, vlines_events)


fig.show()

### Process aggregates

In [11]:
data_payment_value_seg = add_segment_transition_columns(data, 'payer_value_segment')
data_payment_value_movements_agg = get_segment_movements_agg('payer_value_segment', data_payment_value_seg)
data_payment_value_population_agg = get_segment_population_agg('payer_value_segment', data_payment_value_seg, data_payment_value_movements_agg)
#data_payment_value_population_agg

In [12]:
data_payment_frequency_seg = add_segment_transition_columns(data, 'payment_frequency_segment')
data_payment_frequency_movements_agg = get_segment_movements_agg('payment_frequency_segment', data_payment_frequency_seg)
data_payment_frequency_population_agg = get_segment_population_agg('payment_frequency_segment', data_payment_frequency_seg, data_payment_frequency_movements_agg)
#data_payment_frequency_population_agg

In [13]:
data_do_avg_seg = add_segment_transition_columns(data, 'daily_offers_avg_segment')
data_do_avg_movements_agg = get_segment_movements_agg('daily_offers_avg_segment', data_do_avg_seg)
data_do_avg_population_agg = get_segment_population_agg('daily_offers_avg_segment', data_do_avg_seg, data_do_avg_movements_agg)
#data_do_avg_population_agg

## Population

### Daily Offers segment

In [14]:
plot_segment_population_charts(
    data_do_avg_population_agg,
    'daily_offers_avg_segment',
    vlines_events=vlines_events,
    title_prefix='Daily Offers Avg Segment',
    show_total=False,
    show_share=True,
    height=600,
    width=1200)

### Payment value segment

In [15]:
plot_segment_population_charts(
    data_payment_value_population_agg,
    'payer_value_segment',
    vlines_events=vlines_events,
    title_prefix='Payer Value Segment',
    show_total=False,
    show_share=True,
    height=600,
    width=1200)

## Movements

`metrics=[...]` is the only control over which charts get drawn — pass `metrics=[]` to skip charts entirely. Each entry is either one of 4 dual-line (upgrades vs downgrades) specials, or a single-metric column suffix (one line per tier). Default: `('inflow_counts', 'inflow_shares', 'inflow_ratio_downgrade_to_upgrade', 'inflow_direction_normalized')`.

Dual-line specials:

- `inflow_counts`
- `inflow_shares`
- `outflow_counts`
- `outflow_shares`

Single-metric column suffixes, alphabetically:

- `inflow_direction`
- `inflow_direction_normalized`
- `inflow_downgrades_count`
- `inflow_downgrades_share`
- `inflow_ratio_downgrade_to_upgrade`
- `inflow_ratio_upgrade_to_downgrade`
- `inflow_upgrades_count`
- `inflow_upgrades_share`
- `net_flow`
- `outflow_direction`
- `outflow_direction_normalized`
- `outflow_downgrades_count`
- `outflow_downgrades_share`
- `outflow_ratio_downgrade_to_upgrade`
- `outflow_ratio_upgrade_to_downgrade`
- `outflow_upgrades_count`
- `outflow_upgrades_share`
- `shift_active_days_mean`
- `shift_real_days_mean`

`direction` is the raw signed net headcount (upgrades minus downgrades); `direction_normalized` is the same thing scaled to [-1, 1] by this tier's own total movers that (day, tier) — not an independent metric, just `direction` normalized (computed straight from counts, unaffected by how shares below are normalized).

**`upgrades_share`/`downgrades_share` are each their own 100% breakdown, not one combined pie** (2026-08-20 fix): `upgrades_share` is a tier's cut of *that day's total upgrade volume* across all tiers, and `downgrades_share` is separately a tier's cut of *that day's total downgrade volume* — each sums to exactly 1.0 across tiers per day, independently. They used to share one denominator (total movers across both directions), which meant a structurally one-sided tier (e.g. the lowest tier, which only ever receives downgrades) diluted every other tier's *upgrades_share* even though it contributed zero upgrades. Splitting the denominator by direction fixes that without excluding any tier — the lowest tier still correctly shows a large `downgrades_share` (it really is the dominant downgrade destination), it just no longer drags down other tiers' `upgrades_share`.

Not chartable as a line: `inflow_winner` / `outflow_winner` (text values, not numbers).

Removed (2026-08-20): `jump_size_mean`, `x_times_bigger`, `log_ratio_u_to_d` (low-value/unused); `share_ratio` (was an exact duplicate of `ratio_downgrade_to_upgrade` despite its name — never actually share-based); `inflow_ratio` (cruder, sign-carrying, uncorrected predecessor of `ratio_downgrade_to_upgrade`, superseded by it); `net_share` (inflow_direction divided by the day's total movers across all tiers — a mixed denominator that didn't read as intuitive next to direction_normalized's per-tier normalization).

### Daily Offers

#### Average value

In [16]:
plot_movement_charts(
    data_do_avg_movements_agg, 
    'daily_offers_avg_segment', 
    vlines_events=vlines_events, 
    arpdau_df=data_arpdau_agg, 
    overlay_arpdau=False, 
    title_prefix='Daily Offers Avg Segment', 
    metrics=['net_flow','inflow_shares','outflow_shares'],
    height=600,
    width=1200)

### Payment value

In [17]:
plot_movement_charts(
    data_payment_value_movements_agg, 
    'payer_value_segment', 
    vlines_events=vlines_events, 
    arpdau_df=data_arpdau_agg, 
    overlay_arpdau=False, 
    title_prefix='Payer Value Segment', 
    metrics=['net_flow','inflow_shares','outflow_shares'],
    height=600,
    width=1200)

#plot_movement_charts(data_payment_value_movements_agg, 'payer_value_segment', vlines_events=vlines_events, arpdau_df=data_arpdau_agg)

### Payment frequency

In [18]:
plot_movement_charts(
    data_payment_frequency_movements_agg, 
    'payment_frequency_segment', 
    vlines_events=vlines_events, 
    arpdau_df=data_arpdau_agg, 
    overlay_arpdau=False, 
    metrics=['net_flow','inflow_shares','outflow_shares'],
    height=600,
    width=1200)


## Performance Summary

Period-over-period view combining population, `net_flow` (scaled to % of that tier's own population, so growth is comparable across differently-sized tiers), and the inflow/outflow downgrade-to-upgrade ratios — one row per (period, tier). Built because an all-time average can hide a trend reversal: e.g. `daily_offers_avg_segment`'s `4.00-7.99` tier looks mildly positive over the whole window, but was +0.12% in July and -0.575% in August, while its `outflow_ratio` had been climbing every month all along.

`daily_offers_avg_segment` is a 30-day rolling-window segment, so its first 30 days are a ramp artifact (see Population section note above) — excluded here via `min_dt`. `payer_value_segment`/`payment_frequency_segment` use direct SCD joins and don't have that artifact, so no exclusion needed for those.

The two ratio columns are structurally extreme at edge tiers (lowest tier can never have upgrades-in, highest can never have downgrades-in) — expect large/near-zero values there that don't reflect a real trend.

### Daily Offers Avg Segment

In [ ]:
sc = 'daily_offers_avg_segment'
ramp_cutoff = pd.to_datetime(data_do_avg_movements_agg['dt']).min() + pd.Timedelta(days=30)
do_avg_movements_post_ramp = data_do_avg_movements_agg[pd.to_datetime(data_do_avg_movements_agg['dt']) >= ramp_cutoff]
do_avg_population_post_ramp = data_do_avg_population_agg[pd.to_datetime(data_do_avg_population_agg['dt']) >= ramp_cutoff]

do_avg_performance_monthly = get_segment_performance_summary(sc, do_avg_movements_post_ramp, do_avg_population_post_ramp, freq='M')
do_avg_performance_monthly.round(3).pivot(index=sc, columns='period', values='net_flow_pct_of_pop')

In [ ]:
do_avg_performance_weekly = get_segment_performance_summary(sc, do_avg_movements_post_ramp, do_avg_population_post_ramp, freq='W')
plot_segment_performance_summary(do_avg_performance_weekly, sc, title_prefix='Daily Offers Avg Segment')

### Payer Value Segment

In [ ]:
sc = 'payer_value_segment'
payment_value_performance_monthly = get_segment_performance_summary(sc, data_payment_value_movements_agg, data_payment_value_population_agg, freq='M')
payment_value_performance_monthly.round(3).pivot(index=sc, columns='period', values='net_flow_pct_of_pop')

In [ ]:
payment_value_performance_weekly = get_segment_performance_summary(sc, data_payment_value_movements_agg, data_payment_value_population_agg, freq='W')
plot_segment_performance_summary(payment_value_performance_weekly, sc, title_prefix='Payer Value Segment')

### Payment Frequency Segment

In [ ]:
sc = 'payment_frequency_segment'
payment_frequency_performance_monthly = get_segment_performance_summary(sc, data_payment_frequency_movements_agg, data_payment_frequency_population_agg, freq='M')
payment_frequency_performance_monthly.round(3).pivot(index=sc, columns='period', values='net_flow_pct_of_pop')

In [ ]:
payment_frequency_performance_weekly = get_segment_performance_summary(sc, data_payment_frequency_movements_agg, data_payment_frequency_population_agg, freq='W')
plot_segment_performance_summary(payment_frequency_performance_weekly, sc, title_prefix='Payment Frequency Segment')